# 🛰️ Satellite Soil Dataset Builder from LUCAS Dataset
### Pairing LUCAS Ground-Truth Soil Samples with Copernicus Sentinel-2 Multi-Spectral Imagery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### 📌 Purpose:
This notebook extracts Copernicus Sentinel-2 satellite reflectance and calculates spectral indices for soil points from the **LUCAS Soil Dataset** (`LUCAS_Soil_2018.csv`) to create a paired dataset.

**Pipeline Steps:**
1. 🔍 **Search** Microsoft Planetary Computer STAC catalog for cloud-free Sentinel-2 L2A scenes around each soil survey date.
2. ⚡ **Extract** pixel reflectance for all **10 science bands** (`B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12`) and the Scene Classification Layer (`SCL`).
3. 🧪 **Calculate 7 spectral indices** ($BSI, CMR, FII, CI, NDRE, NDVI, SBI$).
4. 💾 **Export** the final merged dataset to CSV.


## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install -q pystac-client planetary-computer rasterio tqdm pandas numpy
print("✅ Libraries installed successfully!")


## 🔌 Step 2: Initialize Planetary Computer STAC Catalog

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from pystac_client import Client
import planetary_computer as pc
import rasterio
from rasterio.windows import Window
from rasterio.warp import transform

# Connect to STAC Catalog
print("Connecting to Planetary Computer STAC API...")
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)
print("✅ STAC Catalog connected successfully!")


## 📂 Step 3: Load LUCAS Soil Dataset
Upload **`LUCAS_Soil_2018.csv`** to the Colab files panel on the left. If the file is not found, an error will be raised.


In [ ]:
# Path to the LUCAS dataset
lucas_filename = "LUCAS_Soil_2018.csv"

# Check current directory and /content/ directory
if os.path.exists(lucas_filename):
    lucas_path = lucas_filename
elif os.path.exists(os.path.join("/content", lucas_filename)):
    lucas_path = os.path.join("/content", lucas_filename)
else:
    raise FileNotFoundError(
        f"❌ ERROR: '{lucas_filename}' was not found in the Colab session!\n"
        f"Please upload '{lucas_filename}' to the Files panel on the left before running this step."
    )

print(f"Loading LUCAS dataset from: {lucas_path}")
df_soil = pd.read_csv(lucas_path)
print(f"✅ Successfully loaded {len(df_soil)} rows.")

# Standardize coordinate and date column names
if 'TH_LAT' in df_soil.columns:
    df_soil['Latitude'] = df_soil['TH_LAT']
if 'TH_LONG' in df_soil.columns:
    df_soil['Longitude'] = df_soil['TH_LONG']
if 'SURVEY_DATE' in df_soil.columns:
    df_soil['Survey_Date'] = pd.to_datetime(df_soil['SURVEY_DATE']).dt.strftime('%Y-%m-%d')
if 'pH_CaCl2' in df_soil.columns:
    df_soil['pH'] = df_soil['pH_CaCl2']

# Display first few rows
display_cols = [c for c in ['POINTID', 'Latitude', 'Longitude', 'Survey_Date', 'pH', 'N', 'P', 'K', 'LC0_Desc'] if c in df_soil.columns]
display(df_soil[display_cols].head(5))


## 🛰️ Step 4: Sentinel-2 Extraction & Spectral Index Functions

In [ ]:
SCIENCE_BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]

def extract_pixel_at_coord(asset_url: str, lon: float, lat: float) -> float:
    """
    Extracts surface reflectance value of a pixel at (lon, lat)
    directly from Cloud-Optimized GeoTIFF.
    Returns np.nan if coordinate is out-of-bounds, nodata, or unreadable.
    """
    try:
        with rasterio.open(asset_url) as src:
            # Transform lon/lat into dataset native projection
            xs, ys = transform('EPSG:4326', src.crs, [lon], [lat])
            x_utm, y_utm = xs[0], ys[0]
            
            # Map coordinates to pixel row and column
            row, col = src.index(x_utm, y_utm)
            
            # Bounds validation
            if row < 0 or row >= src.height or col < 0 or col >= src.width:
                return np.nan
                
            # Read single pixel window
            win = Window(col, row, 1, 1)
            data = src.read(1, window=win)
            
            if data is not None and data.size > 0:
                val = float(data[0, 0])
                if src.nodata is not None and val == src.nodata:
                    return np.nan
                if val <= 0:
                    return np.nan
                return val
            return np.nan
    except Exception:
        return np.nan

def process_single_soil_point(lat: float, lon: float, survey_date_str: str = "2018-07-01", window_days: int = 45):
    """
    Queries Sentinel-2 around survey_date, extracts 10 bands, and calculates spectral indices.
    Iterates candidate scenes to handle points near tile boundaries.
    """
    try:
        center_date = pd.to_datetime(survey_date_str)
        start_date = (center_date - pd.Timedelta(days=window_days)).strftime("%Y-%m-%d")
        end_date = (center_date + pd.Timedelta(days=window_days)).strftime("%Y-%m-%d")
        date_range = f"{start_date}/{end_date}"
    except Exception:
        date_range = "2018-05-01/2018-09-30"

    delta = 0.005  # ~500m bounding box
    search_bbox = [lon - delta, lat - delta, lon + delta, lat + delta]
    
    # 1. Search cloud-free scenes
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=search_bbox,
        datetime=date_range,
        query={"eo:cloud_cover": {"lt": 20}},
        sortby=[{"field": "properties.datetime", "direction": "desc"}]
    )
    items = list(search.items())
    
    # Fallback to wider window if no scene found
    if not items:
        search = catalog.search(
            collections=["sentinel-2-l2a"],
            bbox=search_bbox,
            datetime="2018-04-01/2018-10-31",
            query={"eo:cloud_cover": {"lt": 30}},
            sortby=[{"field": "properties.datetime", "direction": "desc"}]
        )
        items = list(search.items())
        if not items:
            return None

    # 2. Iterate candidate scenes (handles tile border/nodata edges)
    for selected_item in items[:3]:
        scene_date = selected_item.datetime.strftime("%Y-%m-%d")
        cloud_cover = selected_item.properties.get("eo:cloud_cover", 0)
        
        band_values = {}
        valid_scene = True
        
        for band in SCIENCE_BANDS:
            url = selected_item.assets[band].href
            val = extract_pixel_at_coord(url, lon, lat)
            if np.isnan(val):
                valid_scene = False
                break
            band_values[band] = val / 10000.0  # Convert DN to BOA surface reflectance (0-1)
            
        if not valid_scene:
            continue  # Try next candidate scene
            
        # Scene Classification Layer (SCL)
        scl_url = selected_item.assets["SCL"].href
        scl_val = extract_pixel_at_coord(scl_url, lon, lat)
        if np.isnan(scl_val):
            continue
            
        band_values["SCL"] = int(scl_val)
        band_values["S2_SCENE_DATE"] = scene_date
        band_values["S2_CLOUD_COVER"] = cloud_cover
        
        # 3. Calculate Spectral Indices
        b2, b3, b4 = band_values["B02"], band_values["B03"], band_values["B04"]
        b5, b8, b11, b12 = band_values["B05"], band_values["B08"], band_values["B11"], band_values["B12"]
        eps = 1e-6
        
        band_values["NDVI"] = (b8 - b4) / (b8 + b4 + eps)
        band_values["BSI"] = ((b11 + b4) - (b8 + b2)) / ((b11 + b4) + (b8 + b2) + eps)
        band_values["Clay_Ratio"] = b11 / (b12 + eps)
        band_values["Ferric_Iron"] = b4 / (b2 + eps)
        band_values["Carbonate_Index"] = b12 / (b11 + eps)
        band_values["NDRE"] = (b8 - b5) / (b8 + b5 + eps)
        band_values["Soil_Brightness"] = np.sqrt((b4**2 + b3**2 + b8**2) / 3.0)
        
        return band_values

    return None

print("✅ Feature extraction functions ready.")


## 🚀 Step 5: Execute Dataset Generation Pipeline
Set `N_SAMPLES` to the number of rows from `df_soil` you want to process (e.g. 50, 200, or `len(df_soil)` for all).


In [ ]:
# Set number of samples to process
N_SAMPLES = min(50, len(df_soil))
df_to_process = df_soil.head(N_SAMPLES).copy()

results = []
print(f"Processing {len(df_to_process)} soil samples from LUCAS dataset...")

for idx, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Extracting Sentinel-2"):
    lat = float(row['Latitude'])
    lon = float(row['Longitude'])
    date_str = str(row.get('Survey_Date', '2018-07-01'))
    
    try:
        sat_data = process_single_soil_point(lat, lon, date_str)
        if sat_data is not None:
            combined = row.to_dict()
            combined.update(sat_data)
            results.append(combined)
        else:
            print(f"  [!] Skipped sample {idx + 1}: Lat {lat:.4f}, Lon {lon:.4f} (Edge pixel or cloudy)")
    except Exception as e:
        print(f"  [!] Skipped sample {idx + 1} due to error: {e}")

df_final_dataset = pd.DataFrame(results)
print(f"\n✅ Successfully processed {len(df_final_dataset)} samples with Sentinel-2 features!")
display(df_final_dataset.head(5))


## 💾 Step 6: Export and Download Generated Dataset

In [ ]:
OUTPUT_CSV = "lucas_sentinel2_soil_dataset.csv"
df_final_dataset.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved generated dataset to: {OUTPUT_CSV}")
print(f"   Rows: {len(df_final_dataset)}, Columns: {len(df_final_dataset.columns)}")

# Download directly in Google Colab:
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print("Download prompt initiated.")
except ImportError:
    pass
